# BamiBERT LoRA với lọc cụm trùng kiểu

Biến thể giải mã tái sử dụng đúng trọng số của notebook 01. Chọn lại ngưỡng trên validation khi bật lọc.

[Báo cáo tổng hợp](../reports/tong-hop-pipeline-smartnotice-v3.3-2026-11-23.md) · [Giao thức đánh giá](../../../docs/evaluation.md)

## 1. Cấu hình và dữ liệu

Nạp thông báo đã chuẩn bị và kiểm tra hash, nhóm, schema và các khoảng ký tự. Tham chiếu test v3.3 là chú giải lịch sử có máy hỗ trợ.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/compare_notice_pipelines.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Mở notebook từ workspace SmartNotice.")
os.chdir(repo)
split = Path(os.environ.get("SMARTNOTICE_SPLIT_ROOT", "tmp/drive-run-20261123-fresh")).resolve()
base = Path(os.environ.get("SMARTNOTICE_BAMIBERT_DIR", "tmp/bamibert-checkpoint")).resolve()
destination = Path(os.environ.get(
    "SMARTNOTICE_NOTEBOOK_OUTPUT", "reports/generated/pipeline-notebooks-v33-20261123"
)).resolve()
baseline = Path(os.environ.get(
    "SMARTNOTICE_BASELINE", "reports/generated/bamibert/v3_3-2026-11-23-final-seed42"
)).resolve()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_tool(script, *arguments, **environment):
    env = {**os.environ, "PYTHONHASHSEED": "42", "TOKENIZERS_PARALLELISM": "false", **environment}
    result = subprocess.run(
        [sys.executable, "-u", str(repo / "tools" / script), *map(str, arguments)],
        cwd=repo, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:])
        result.check_returncode()
    print(result.stdout.strip())

In [2]:
from smartnotice.modeling.data import labeled_links, load_development, sha256, span_key

development, provenance = load_development(split)
for part, docs in development.items():
    print(part, {
        "notices": len(docs),
        "groups": len({d.group_id for d in docs}),
        "typed_spans": sum(len({span_key(s) for s in d.annotation.spans}) for d in docs),
        "links": sum(len(set(labeled_links(d.annotation))) for d in docs),
        "events": sum(len(d.annotation.events) for d in docs),
    })
print("Phân bổ:", provenance["split_counts"], "gold_ready:", provenance["gold_ready"])

train {'notices': 403, 'groups': 281, 'typed_spans': 4664, 'links': 3847, 'events': 1516}
validation {'notices': 93, 'groups': 77, 'typed_spans': 1078, 'links': 873, 'events': 357}
Phân bổ: {'train': 403, 'test': 126, 'validation': 93, 'domain_test': 62} gold_ready: False


## 2. Nạp trọng số đã học và chọn ngưỡng

Đặt SMARTNOTICE_BASELINE tới thư mục huấn luyện của notebook 01 để so sánh cùng checkpoint.

In [3]:
import shutil
from dataclasses import replace

import torch
from transformers import AutoTokenizer
from tools.compare_notice_pipelines import _bert_predictions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(base, use_fast=True)

out = destination / "run-02"
out.mkdir(parents=True, exist_ok=False)
app_run = baseline

record = read_json(app_run / "run.json")
assert record["data"]["hashes"] == provenance["hashes"]
assert record["model_weights_sha256"] == sha256(base / "model.safetensors")
assert record["tokenizer_sha256"] == sha256(base / "tokenizer.json")

for name in ("run.json", "extraction.pt", "applicability.pt"):
    shutil.copyfile(app_run / name, out / name)
run_tool("evaluate_bamibert_decoder.py", "--split", split, "--checkpoint", base, "--run", out)
extraction = read_json(out / "overlap_suppression_metrics.json")["extraction"]
print("Ngưỡng validation:", extraction["thresholds"])
print("F1 validation:", extraction["span"]["micro"]["f1"], extraction["link_predicted_endpoints"]["micro"]["f1"])
weights = out / "suppressed.pt"
state = torch.load(out / "extraction.pt", map_location="cpu")
state["settings"]["suppress_same_type_overlap"] = True
state["thresholds"] = extraction["thresholds"]
torch.save(state, weights)

Replayed both checkpoints and saved overlap suppression metrics
Ngưỡng validation: {'span': 0.9, 'link': 0.5, 'oracle_link': 0.5}
F1 validation: 0.11067193675889328 0.004730368968779565


## 3. Suy luận test với bộ lọc đã chốt và chấm điểm

In [4]:
from smartnotice.modeling.experiment import write_summary
from smartnotice.modeling.metrics import prf


def show(scores):
    span = scores["span"]["micro"]
    link = scores["link"]["micro"]
    event = scores["event"]
    for caption, score in (("Cụm có kiểu", span), ("Quan hệ", link), ("Sự kiện", event)):
        print(
            f"{caption}: F1 {score['f1']:.2%} | P {score['precision']:.2%} | "
            f"R {score['recall']:.2%} | đúng/dự đoán/tham chiếu "
            f"{score['matched']}/{score['predicted']}/{score['expected']}"
        )
    print("Ngày tường minh:", scores["dates"]["explicit"])


def without_reference(docs):
    from smartnotice.modeling.data import Document
    from smartnotice.schemas import Annotation

    return [
        Document(d.notice, Annotation(
            doc_id=d.notice.doc_id, annotator_id="inference", spans=[], events=[]
        ), d.group_id)
        for d in docs
    ]


def boundary_diagnostics(docs, predictions):
    expected, predicted = {}, {}
    for d in docs:
        doc_id = d.notice.doc_id
        for target, spans in (
            (expected, map(span_key, d.annotation.spans)), (predicted, predictions[doc_id][0])
        ):
            for begin, end, kind in spans:
                target.setdefault((doc_id, begin, end), set()).add(kind)
    matched = expected.keys() & predicted.keys()
    return {
        "untyped": prf(set(expected), set(predicted)),
        "correct_type_sets": sum(expected[key] == predicted[key] for key in matched),
        "matched_boundaries": len(matched),
    }

In [5]:
from tools.compare_notice_pipelines import _metrics, _test_docs

test = _test_docs(split)
print("Test:", len(test), "thông báo,", len({d.group_id for d in test}), "nhóm")

Test: 126 thông báo, 118 nhóm


In [6]:
predictions = _bert_predictions(weights, without_reference(test), base, tokenizer, device)
result = _metrics(test, predictions)
result["boundaries"] = boundary_diagnostics(test, predictions)
show(result)
print("Ranh giới và gán kiểu có điều kiện:", result["boundaries"])

Cụm có kiểu: F1 12.52% | P 14.21% | R 11.19% | đúng/dự đoán/tham chiếu 158/1112/1412
Quan hệ: F1 0.16% | P 0.14% | R 0.17% | đúng/dự đoán/tham chiếu 2/1429/1149
Sự kiện: F1 0.00% | P 0.00% | R 0.00% | đúng/dự đoán/tham chiếu 0/206/468
Ngày tường minh: {'expected_links': 188, 'matched_links': 0, 'correct_dates': 0, 'matched_date_accuracy': None, 'matched_inferred_flag_accuracy': None, 'end_to_end_date_accuracy': 0.0}
Ranh giới và gán kiểu có điều kiện: {'untyped': {'precision': 0.1420863309352518, 'recall': 0.11189801699716714, 'f1': 0.12519809825673534, 'matched': 158, 'predicted': 1112, 'expected': 1412}, 'correct_type_sets': 158, 'matched_boundaries': 158}


In [7]:
from types import SimpleNamespace
from tools.compare_notice_pipelines import _app

app = _app(
    SimpleNamespace(base=base, baseline=app_run),
    development["train"], test, {"pipeline": predictions}, device, tokenizer,
)["pipeline"]
result["applicability"] = app
print("F1 sự kiện và nhãn áp dụng:", f"{app['joint_exact_event_profile']['f1']:.2%}")
write_summary(out / "test_scores.json", result)

F1 sự kiện và nhãn áp dụng: 0.00%
